# MIRI Drug Repositioning Screen — xpert-fda-miri-v1.1

DToxS-LINCS testbed sibling of the finalized xpert-fda-miri-v1 screen. Only FDA drugs present in the L1000 embedding table (exact SMILES match, drug_idx < 8981) are screened; the HG biological embeddings are the **author reference table** — no HG training / FDA-extension inference. The XPert finetune (steps 05/06) trains on the workflow-rebuilt **DToxS-LINCS h5ad** (step 03, from the released LINCS LDS-1593 well-level package at `pert_data/dtoxs_primary/Data/Level3`), not v1's frozen `input/dtoxs.h5ad`. v1 stays untouched. Run cells in order. Environment: `conda activate arc`. Heavy steps (05 benchmark, 06 finetune, 07 predict) require a GPU.

## Step 01 — Drug list + embeddings

Filters the ChEMBL FDA table (`input/fda_drugs.csv`) to drugs whose exact canonical SMILES is in the L1000 rows of the embedding mapping (`drug_idx < 8981`), then copies the author reference HG table (workflow-owned `input/HG_drug_embeddings.npy`) to `01_drug_embed/output/all_drugs_idx2HG.npy` (byte-identical parity gate). Outputs: `01_drug_embed/output/drug_list.csv` (1,004 drugs) + `all_drugs_idx2HG.npy`.

In [ ]:
!python 01_drug_embed/drug_list.py
!python 01_drug_embed/drug_embeddings.py

## Gene mapping — DToxS-primary -> L1000 + verification

Builds the DToxS-primary landmark map (symbol space, resolving the 18 legacy-symbol landmarks WASHC5<-KIAA0196, JPT2<-HN1L, ...) and gates all 978 landmarks. Run before steps 02/03.

In [ ]:
!python _lib/verify_genes.py

## Step 02 — CM baseline transcriptome

Builds the consensus cardiomyocyte baseline from the DToxS-primary control wells of the released LINCS LDS-1593 package (`pert_data/dtoxs_primary/Data/Level3`; the raw GEO deposit was deleted 2026-08-18) using the authoritative gene map. Identical to v1's baseline. Output: `02_cm_baseline/output/cm_ctl_landmark.csv`.

In [ ]:
!python 02_cm_baseline/cm_baseline.py

## Step 03 — DToxS-LINCS finetune data (the v1.1 change)

Rebuilds the DToxS finetune h5ad `03_cm_finetune_data/output/dtoxs_lincs.h5ad` from the released LINCS LDS-1593 well-level package (`pert_data/dtoxs_primary/Data/Level3`, read-only; the deleted raw GEO deposit lacked the non-KI perturbations), **per-plate normalized** since ADR 0016 (QN + matched control per `(dataset, plate)`) + the drug manifest and the parity delta vs v1's frozen `input/dtoxs.h5ad`. The parity gate requires schema parity (fields the XPert loader consumes) and reports the sample/drug delta. Also generates the composition report `finetune_report.html`.

In [ ]:
!python 03_cm_finetune_data/build_dtoxs_lincs.py
!python 03_cm_finetune_data/finetune_report.py

## Step 04 — XPert pretrain (reference, workflow-owned)

Holds the author pretrained XPert base weight (`pretrain_mdmt_full_50_epoch.pth`) and the L1000 pretrain YAML that produced it — the pretrain dataset lives under `pert_data/l1000-xpert/`. Not re-run during screening; steps 05/06 consume the weight from here.

In [ ]:
!ls 04_xpert_pretrain/

## Step 05 — Benchmark XPert on the DToxS-LINCS splits

Creates the unified fold columns (warm-start + cold-drug, 10 folds each, ADR 0014) + `dtoxs_full` (all-train) on the rebuilt h5ad and finetunes each fold from the pretrain weight, evaluating on the held-out test set. The L1000 pretrain weight is also evaluated zero-shot (no finetuning) on all 20 fold test sets and plotted as the FIRST column (`pretrain`, amber). Per-fold values are overlaid as black dots on the mean ± std bars. Outputs: `05_xpert_finetune_bench/output/benchmark.csv` + `benchmark.png`.

In [ ]:
!python 05_xpert_finetune_bench/create_dtoxs_splits.py
!python 05_xpert_finetune_bench/xpert_finetune_bench.py

## Step 06 — Finetune XPert on the DToxS-LINCS data (dtoxs_full)

All-train DToxS-LINCS finetune -> the final prediction model (base weight from step 04). Idempotent (weights kept). Output: `06_xpert_finetune/output/dtoxs_dtoxs/`.

In [ ]:
!python 06_xpert_finetune/xpert_finetune.py

## Step 07 — Perturbed transcriptome + z-score

Predicts drug perturbation (deg) for the 1,004 L1000-contained FDA drugs via XPert (step-06 model), then z-score normalizes the DEG per gene across drugs. Predict indexes the precomputed UniMol array by drug index, so finetune and predict consume identical features. Outputs: `07_xpert_predict/output/xpert_fda-cm_deg.csv` + `-zs.csv`.

In [ ]:
!python 07_xpert_predict/perturb_transcriptome.py
!python 07_xpert_predict/zscore.py

## Step 08 — MIRI reference gene sets + pathways

Consumes the per-GSE DEG lists in `disease_data/miri/{GSE}/deg/` and produces the integrated reference (`MIRI_up-down.json`, `MIRI_rank.json`). `input/MIRI_pathways.json` is static (curated).

In [ ]:
!python 08_miri_phenotype/miri_phenotype.py

## Step 09 — Efficacy scores

Runs the three scorers (WCS+RGES, Zhang/ssCMap, pathway RGES) and computes the z-components for all screened drugs, plus the score-distribution plots. Output: `09_miri_efficacy_scores/output/scores.csv` + PNGs.

In [ ]:
!python 09_miri_efficacy_scores/miri_efficacy_scores.py

## Step 10 — Filter

Applies the WCS<0 and >=6/8 critical-pathway filters + filter-progression pyramid plot. Outputs: `10_filter/output/wcs_filtered.csv`, `path_filtered.csv`, `pyramid-filters.png`.

In [ ]:
!python 10_filter/filter.py

## Step 11 — Integrate

Computes the integrated score over all screened drugs, ranks the filtered candidates, and writes the final report: `11_integrate/output/integrated_scores.csv`, `ranked_drugs.csv`, `report.md` + rank heatmaps.

In [ ]:
!python 11_integrate/integrate.py

## Pipeline summary

The final ranked drug list is `11_integrate/output/ranked_drugs.csv` with the deterministic report at `11_integrate/output/report.md`. v1.1 has **no agentic report stage** (the former `12_report` was removed per spec 1; v1 keeps its frozen `11_report`).

In [ ]:
import pandas as pd
ranked = pd.read_csv('11_integrate/output/ranked_drugs.csv')
print(f'{len(ranked)} drugs passed the filters')
print(ranked[['drug_name', 'integrated']].head(10).to_string(index=False))